In [28]:
# read gift.csv and print the contents by numpy pandas and csv module
import csv
import numpy as np
import pandas as pd
data_ground = pd.read_csv('gift.csv')
# data in columns 0 replace string - to _
data_ground.iloc[:, 0] = data_ground.iloc[:, 0].str.replace('-', '_')
# data_ground

In [29]:
data = pd.read_csv('rgb_augmented.csv')
# show ID = "M_136_1"
# data[data['ID'] == "M_136_1"]

In [30]:
# mearge data_ground and data by column 0
data_merged = pd.merge(data_ground, data, on=data_ground.columns[0])
# data_merged

In [31]:
data_merged.drop_duplicates(inplace=True)
data_merged.rename(columns={"R_mean": 'R', "G_mean": 'G', "B_mean": 'B'}, inplace=True)
data_merged.rename(columns={"ID": 'VDO'}, inplace=True)

# data_merged

In [32]:
# import library RGB to Lab
from skimage import color
import numpy as np

from skimage import color
import numpy as np

def bt709_to_linear(c):
    return np.where(c < 0.081, c / 4.5, ((c + 0.099) / 1.099) ** (1 / 0.45))

def calculate_lab(row):
    r, g, b = row['R'] / 255.0, row['G'] / 255.0, row['B'] / 255.0
    rgb_linear = bt709_to_linear(np.array([r, g, b]))
    # แปลง linear RGB → XYZ ด้วย BT.709/sRGB matrix
    M = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    xyz = M @ rgb_linear
    # XYZ → Lab (D65)
    lab_pixel = color.xyz2lab(xyz.reshape(1,1,3), illuminant='D65')
    L, a, b = lab_pixel[0,0]
    return round(L, 2), round(a, 2), round(b, 2)

# convert RGB to Lab และสร้างคอลัมน์ใหม่เพื่อเทียบกับค่าเฉลย
data_merged[['L_cal', 'a_cal', 'b_cal']] = data_merged.apply(calculate_lab, axis=1, result_type='expand')

# data_merged

In [33]:
data_merged.drop_duplicates(inplace=True)
data_merged.describe()

,L*,a*,b*,Crop_Index,R,G,B,L_cal,a_cal,b_cal
count,20180.000000,20180.000000,20180.000000,20180.000000,20180.000000,20180.000000,20180.000000,20180.000000,20180.000000,20180.000000
mean,52.074628,4.757473,3.695951,5.500000,125.501060,115.500743,129.713679,56.822861,12.495806,-1.880400
std,21.825987,21.843654,26.051741,2.872352,83.729313,73.547872,77.379506,21.895662,31.974768,40.916323
min,23.610000,-41.160000,-36.160000,1.000000,13.314800,0.000000,14.914900,11.940000,-52.930000,-78.020000
25%,32.710000,-1.530000,-7.760000,3.000000,33.433825,39.316700,34.284525,39.480000,-1.442500,-10.880000
50%,48.610000,4.340000,-0.490000,5.500000,158.819050,131.097950,174.952350,59.360000,5.050000,-2.480000
75%,66.960000,13.840000,14.320000,8.000000,211.559075,174.641450,196.318400,74.710000,41.800000,27.932500
max,96.150000,46.770000,61.840000,10.000000,219.452400,223.007200,225.633500,89.420000,65.630000,63.690000


In [34]:
data_merged.drop(columns=['VDO', 'File_Name', 'Crop_Index'], inplace=False).corr()

,L*,a*,b*,R,G,B,L_cal,a_cal,b_cal
L*,1.000000,-0.090730,0.241382,0.744847,0.853852,0.433020,0.931512,-0.284951,0.301904
a*,-0.090730,1.000000,0.140491,0.469981,-0.544452,-0.226514,-0.261565,0.938795,0.052421
b*,0.241382,0.140491,1.000000,0.605944,0.176847,-0.703845,0.302731,-0.072596,0.945853
R,0.744847,0.469981,0.605944,1.000000,0.411045,-0.060750,0.667335,0.237699,0.602846
G,0.853852,-0.544452,0.176847,0.411045,1.000000,0.473411,0.933558,-0.676169,0.231813
B,0.433020,-0.226514,-0.703845,-0.060750,0.473411,1.000000,0.416071,-0.119420,-0.702405
L_cal,0.931512,-0.261565,0.302731,0.667335,0.933558,0.416071,1.000000,-0.413225,0.350361
a_cal,-0.284951,0.938795,-0.072596,0.237699,-0.676169,-0.119420,-0.413225,1.000000,-0.184461
b_cal,0.301904,0.052421,0.945853,0.602846,0.231813,-0.702405,0.350361,-0.184461,1.000000


In [35]:
data_merged

,VDO,L*,a*,b*,File_Name,Crop_Index,R,G,B,L_cal,a_cal,b_cal
0,M_136_1,34.53,31.53,16.21,M_136_1.mp4_sec_0000.jpg,1,179.9762,1.0246,26.0372,39.36,64.04,33.99
1,M_136_1,34.53,31.53,16.21,M_136_1.mp4_sec_0000.jpg,2,183.2693,22.4903,50.7522,42.49,59.78,22.83
2,M_136_1,34.53,31.53,16.21,M_136_1.mp4_sec_0000.jpg,3,180.8743,0.0499,24.8452,39.42,64.51,34.83
3,M_136_1,34.53,31.53,16.21,M_136_1.mp4_sec_0000.jpg,4,181.0952,0.1012,26.0082,39.48,64.58,34.20
4,M_136_1,34.53,31.53,16.21,M_136_1.mp4_sec_0000.jpg,5,183.2210,0.4137,22.5022,39.89,64.85,36.86
...,...,...,...,...,...,...,...,...,...,...,...,...
20175,M_444_3,90.65,2.02,-7.72,M_444_3.mp4_sec_0030.jpg,6,212.0620,215.0620,220.0736,87.43,-0.08,-2.52
20176,M_444_3,90.65,2.02,-7.72,M_444_3.mp4_sec_0030.jpg,7,216.0000,219.0000,224.0000,88.68,-0.09,-2.51
20177,M_444_3,90.65,2.02,-7.72,M_444_3.mp4_sec_0030.jpg,8,214.6768,219.7720,223.8716,88.77,-0.86,-2.33
20178,M_444_3,90.65,2.02,-7.72,M_444_3.mp4_sec_0030.jpg,9,215.8651,218.8824,223.8564,88.64,-0.10,-2.50


In [36]:
# data_merged to csv
data_merged.to_csv("data.csv")